# Final YAMNet Training

This notebook is fully self-contained for Google Colab and trains the final model on **all rows** from `wolfset_train.csv`.

Use the hyperparameters chosen from the cross-validation notebook, then save the trained classifier and supporting artifacts.


In [1]:
!pip -q install "setuptools<81" tensorflow tensorflow-hub librosa scikit-learn pandas seaborn matplotlib scipy joblib tqdm

In [2]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Google Colab not detected. If you are running locally, update PROJECT_ROOT below.")


PROJECT_ROOT = Path("/content/drive/MyDrive/1:1_Arjan_Walia/Vessels")  # Update this if your Drive path is different.
TRAIN_CSV = PROJECT_ROOT / "datasets" / "processed_files" / "wolfset_train.csv"
AUDIO_DIR = PROJECT_ROOT / "datasets" / "25791978"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts/new"
EMBEDDING_CACHE_DIR = ARTIFACT_DIR / "yamnet_embeddings_train_5s"
TFHUB_CACHE_DIR = ARTIFACT_DIR / "tfhub_cache"
CV_SUMMARY_PATH = ARTIFACT_DIR / "yamnet_cv_summary.csv"
MODEL_PATH = ARTIFACT_DIR / "yamnet_classifier.keras"
SCALER_PATH = ARTIFACT_DIR / "yamnet_scaler.joblib"
LABEL_METADATA_PATH = ARTIFACT_DIR / "label_metadata.json"
TRAINING_CONFIG_PATH = ARTIFACT_DIR / "yamnet_training_config.json"
HISTORY_PATH = ARTIFACT_DIR / "yamnet_training_history.csv"

TARGET_SAMPLE_RATE = 16000
CLIP_DURATION_SECONDS = 5.0
BATCH_SIZE = 16
SEED = 42

# Change this with the best epochs and learning rate combination
SELECTED_LEARNING_RATE = 0.001
SELECTED_EPOCHS = 20

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_CSV.exists(), f"Could not find: {TRAIN_CSV}"
assert AUDIO_DIR.exists(), f"Could not find: {AUDIO_DIR}"

Mounted at /content/drive


In [3]:
import json
import os
import random
from itertools import product
from pathlib import Path

import joblib
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import tensorflow_hub as hub
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAME_MAP = {
    1: "4.5hp",
    2: "18hp",
    3: "8hp",
    4: "3.6hp",
    5: "Electric",
}


def load_split_dataframe(csv_path: Path, audio_dir: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path).copy()
    df["label"] = df["label"].astype(int)
    df["audio_path"] = df["filename"].map(lambda name: str((audio_dir / name).resolve()))
    missing = [name for name, path in zip(df["filename"], df["audio_path"]) if not Path(path).exists()]
    if missing:
        raise FileNotFoundError(f"Missing {len(missing)} audio files. First few: {missing[:5]}")
    return df.reset_index(drop=True)


def load_waveform(audio_path: str, target_sample_rate: int = 16000, clip_duration_seconds: float = 5.0) -> np.ndarray:
    waveform, _ = librosa.load(audio_path, sr=target_sample_rate, mono=True, duration=clip_duration_seconds)
    waveform = waveform.astype(np.float32)
    peak = np.max(np.abs(waveform))
    if peak > 0:
        waveform = waveform / peak
    return waveform


def load_yamnet_model(cache_dir: Path):
    cache_dir.mkdir(parents=True, exist_ok=True)
    os.environ["TFHUB_CACHE_DIR"] = str(cache_dir)
    return hub.load("https://tfhub.dev/google/yamnet/1")


def extract_yamnet_embedding(audio_path: str, yamnet_model, target_sample_rate: int = 16000, clip_duration_seconds: float = 5.0) -> np.ndarray:
    waveform = load_waveform(audio_path, target_sample_rate=target_sample_rate, clip_duration_seconds=clip_duration_seconds)
    _, embeddings, _ = yamnet_model(waveform)
    return np.mean(embeddings.numpy(), axis=0).astype(np.float32)


def build_embedding_matrix(
    metadata: pd.DataFrame,
    yamnet_model,
    cache_dir: Path,
    target_sample_rate: int = 16000,
    clip_duration_seconds: float = 5.0,
) -> tuple[pd.DataFrame, np.ndarray]:
    cache_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    embeddings = []

    for _, row in tqdm(metadata.iterrows(), total=len(metadata), desc="YAMNet embeddings"):
        cache_name = f"{Path(row['filename']).stem}_sr{target_sample_rate}_{clip_duration_seconds:g}s.npy"
        cache_path = cache_dir / cache_name

        if cache_path.exists():
            embedding = np.load(cache_path)
        else:
            embedding = extract_yamnet_embedding(
                audio_path=row["audio_path"],
                yamnet_model=yamnet_model,
                target_sample_rate=target_sample_rate,
                clip_duration_seconds=clip_duration_seconds,
            )
            np.save(cache_path, embedding)

        rows.append(row)
        embeddings.append(embedding.astype(np.float32))

    return pd.DataFrame(rows).reset_index(drop=True), np.vstack(embeddings).astype(np.float32)


def build_classifier(input_dim: int, num_classes: int, learning_rate: float, dropout_rate: float = 0.30) -> tf.keras.Model:
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dropout(dropout_rate),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(dropout_rate / 2.0),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def compute_class_weight_dict(y_encoded: np.ndarray) -> dict[int, float]:
    classes = np.unique(y_encoded)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_encoded)
    return {int(cls): float(weight) for cls, weight in zip(classes, weights)}


In [4]:
if CV_SUMMARY_PATH.exists():
    cv_summary = pd.read_csv(CV_SUMMARY_PATH).sort_values(by=["mean_macro_f1", "mean_accuracy"], ascending=False)
    display(cv_summary.head())
    best = cv_summary.iloc[0]
    SELECTED_LEARNING_RATE = float(best["learning_rate"])
    SELECTED_EPOCHS = int(best["epochs"])

print({
    "selected_learning_rate": SELECTED_LEARNING_RATE,
    "selected_epochs": SELECTED_EPOCHS,
})

train_df = load_split_dataframe(TRAIN_CSV, AUDIO_DIR)
yamnet_model = load_yamnet_model(TFHUB_CACHE_DIR)
train_df, X = build_embedding_matrix(
    metadata=train_df,
    yamnet_model=yamnet_model,
    cache_dir=EMBEDDING_CACHE_DIR,
    target_sample_rate=TARGET_SAMPLE_RATE,
    clip_duration_seconds=CLIP_DURATION_SECONDS,
)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train_df["label"].to_numpy())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


{'selected_learning_rate': 0.001, 'selected_epochs': 20}


YAMNet embeddings:   0%|          | 0/85 [00:00<?, ?it/s]

In [ ]:
model = build_classifier(
    input_dim=X_scaled.shape[1],
    num_classes=len(label_encoder.classes_),
    learning_rate=SELECTED_LEARNING_RATE,
)

history = model.fit(
    X_scaled,
    y,
    epochs=SELECTED_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
    class_weight=compute_class_weight_dict(y),
)

train_probabilities = model.predict(X_scaled, verbose=0)
train_predictions = train_probabilities.argmax(axis=1)
train_accuracy = (train_predictions == y).mean()

print("Training-set accuracy:", round(float(train_accuracy), 4))
print("Classes used:", dict(zip(label_encoder.classes_, [CLASS_NAME_MAP.get(int(v), f"Class {v}") for v in label_encoder.classes_])))


Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 359ms/step - accuracy: 0.3093 - loss: 1.7867
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8041 - loss: 0.6236 
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9175 - loss: 0.3492 
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9485 - loss: 0.2038 
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9691 - loss: 0.1621 
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9794 - loss: 0.1140 
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9897 - loss: 0.0952 
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9897 - loss: 0.0839 
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0531 
Epoch 10/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0388 
Epoch 11/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9897 - loss: 0.0673 
Epoch 12/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0317 


In [ ]:
model.save(MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)

label_metadata = {
    "classes": [int(value) for value in label_encoder.classes_],
    "class_names": [CLASS_NAME_MAP.get(int(value), f"Class {value}") for value in label_encoder.classes_],
}
LABEL_METADATA_PATH.write_text(json.dumps(label_metadata, indent=2))

history_df = pd.DataFrame(history.history)
history_df.to_csv(HISTORY_PATH, index=False)

TRAINING_CONFIG_PATH.write_text(
    json.dumps(
        {
            "learning_rate": SELECTED_LEARNING_RATE,
            "epochs": SELECTED_EPOCHS,
            "batch_size": BATCH_SIZE,
            "target_sample_rate": TARGET_SAMPLE_RATE,
            "clip_duration_seconds": CLIP_DURATION_SECONDS,
            "num_training_samples": int(len(train_df)),
            "classes": [int(value) for value in label_encoder.classes_],
        },
        indent=2,
    )
)

print("Saved artifacts:")
print(MODEL_PATH)
print(SCALER_PATH)
print(LABEL_METADATA_PATH)
print(TRAINING_CONFIG_PATH)
print(HISTORY_PATH)


Saved artifacts:
/content/drive/MyDrive/1:1_Arjan_Walia/Vessels/artifacts/new/yamnet_classifier.keras
/content/drive/MyDrive/1:1_Arjan_Walia/Vessels/artifacts/new/yamnet_scaler.joblib
/content/drive/MyDrive/1:1_Arjan_Walia/Vessels/artifacts/new/label_metadata.json
/content/drive/MyDrive/1:1_Arjan_Walia/Vessels/artifacts/new/yamnet_training_config.json
/content/drive/MyDrive/1:1_Arjan_Walia/Vessels/artifacts/new/yamnet_training_history.csv
